# Setup do SageMaker Model Monitor

Esse notebook configura o Model Monitor no endpoint do SageMaker.
Precisa rodar uma vez após o deploy — o monitoramento passa a ser automático depois disso.

**Pré-requisitos:**
- Endpoint `passos-magicos-prod` já criado
- Bucket S3 de monitoramento criado pelo CloudFormation
- Credenciais AWS configuradas (`aws configure`)

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import boto3
import sagemaker
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

session = sagemaker.Session()
region = boto3.Session().region_name
account = boto3.client('sts').get_caller_identity()['Account']

ENDPOINT_NAME = "passos-magicos-prod"
BUCKET = f"passos-magicos-monitoring-{account}-prod"

# pega a role do SageMaker via boto3
iam = boto3.client('iam')
role_arn = iam.get_role(RoleName='SageMakerExecutionRole')['Role']['Arn']

print(f"região: {region}")
print(f"bucket: {BUCKET}")
print(f"endpoint: {ENDPOINT_NAME}")
print(f"role: {role_arn}")

## 1. Envia os dados de treino pro S3 (baseline)

O Model Monitor precisa de um dataset de referência para comparar com os dados em produção.

In [ ]:
import pandas as pd
from src.preprocessing import load_raw_data, clean_data, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from src.feature_engineering import create_features, get_extended_feature_columns

df_raw = load_raw_data()
df = clean_data(df_raw)
df = create_features(df)

feature_cols = get_extended_feature_columns(df) + [c for c in CATEGORICAL_FEATURES if c in df.columns]
df_baseline = df[feature_cols].dropna()

# salva localmente e sobe pro S3
baseline_path = ROOT / "database" / "baseline.csv"
df_baseline.to_csv(baseline_path, index=False)

s3 = boto3.client('s3')
s3.upload_file(str(baseline_path), BUCKET, "baseline/baseline.csv")

print(f"baseline enviado: {len(df_baseline)} amostras, {len(feature_cols)} features")

## 2. Cria o baseline

O SageMaker roda um job que calcula as estatísticas e constraints do dataset de referência.
Esse job leva alguns minutos.

In [ ]:
monitor = DefaultModelMonitor(
    role=role_arn,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=session,
)

monitor.suggest_baseline(
    baseline_dataset=f"s3://{BUCKET}/baseline/baseline.csv",
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{BUCKET}/baseline-results/",
    wait=True,
    logs=False,
)

print("baseline criado!")

## 3. Cria o schedule de monitoramento

Configura o Model Monitor pra rodar automaticamente a cada hora e comparar os dados capturados com o baseline.

In [ ]:
from sagemaker.model_monitor import CronExpressionGenerator

monitor.create_monitoring_schedule(
    monitor_schedule_name="passos-magicos-monitor",
    endpoint_input=ENDPOINT_NAME,
    output_s3_uri=f"s3://{BUCKET}/monitoring-results/",
    statistics=monitor.baseline_statistics(),
    constraints=monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
)

print("schedule criado — monitoramento rodando a cada hora")

## 4. Verifica os resultados

Depois que o schedule rodar pelo menos uma vez, é possível ver as violações detectadas.

In [ ]:
execucoes = monitor.list_executions()

if not execucoes:
    print("nenhuma execução ainda — aguarde o próximo ciclo do schedule")
else:
    ultima = execucoes[-1]
    print(f"última execução: {ultima.describe()['ScheduledTime']}")
    print(f"status: {ultima.describe()['MonitoringExecutionStatus']}")

    # mostra violações, se houver
    violations = ultima.constraint_violations()
    if violations and violations.body_dict.get('violations'):
        print(f"\nviolações detectadas:")
        for v in violations.body_dict['violations']:
            print(f"  - {v['feature_name']}: {v['description']}")
    else:
        print("\nnenhuma violação — modelo estável")

## Comandos úteis

```python
# pausar o monitoramento
monitor.stop_monitoring_schedule()

# deletar o schedule
monitor.delete_monitoring_schedule()

# ver no console AWS
# SageMaker > Endpoints > passos-magicos-prod > Model Monitoring
```